#Schema evolution:
  add/drop columns in the existing table/file is allowed<br>
  
#Schema Enforcement:
<br>
  When writing the data into table/file, ensure that data is in existing file/data format

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
from pyspark.sql.functions import to_date

spark = SparkSession.builder.appName("MovieDeltaExample").getOrCreate()

data = [
    ("Inception", "2010-07-16", 8.8),
    ("Interstellar", "2014-11-07", 8.6),
    ("The Dark Knight", "2008-07-18", 9.0),
    ("Tenet", "2020-08-26", 7.5),
    ("Oppenheimer", "2023-07-21", 8.7),
    ("Dune", "2021-10-22", 8.0),
    ("Avatar", "2009-12-18", 7.8),
    ("The Matrix", "1999-03-31", 8.7)
]

schema = StructType([
    StructField("title", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("rating", DoubleType(), True)
])

df = spark.createDataFrame(data, schema)
df = df.withColumn("release_date", to_date("release_date"))

df.show()


In [0]:
df.write.format("delta").mode("overwrite").save("dbfs:/Volumes/databricks_practice/inputdb/moviesdata/movies_delta")


In [0]:
load_path="dbfs:/Volumes/databricks_practice/inputdb/moviesdata/movies_delta"
spark.read.format("delta").load(load_path).display()

In [0]:
load_path="dbfs:/Volumes/databricks_practice/inputdb/moviesdata/movies_delta"
spark.sql(f"select * from delta.`{load_path}`").display()

In [0]:
spark.sql(f"select * from delta.`{load_path}`").printSchema()

In [0]:

### whenever there schmae change like column name change delta format will not allow it for processing unlike other format like csv
bad_data = [
    ("Thalapathi", "1992-07-16", 8.8),
    ("Muthu", "1992-07-16", 8.8)
]

schema = StructType([
    StructField("movietitle", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("rating", DoubleType(), True)
])

bad_df = spark.createDataFrame(bad_data, schema)
bad_df = bad_df.withColumn("release_date", to_date("release_date"))

bad_df.show()

bad_df.write.format("delta").mode("append").save(load_path)
